In [1]:
import pandas as pd
import nltk
import spacy
import re

from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag, word_tokenize

In [2]:
# 1. Download NLTK Resources
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is

True

In [3]:
# 2. Load Dataset
INPUT_FILE = "fakenews_preprocessed.csv"
OUTPUT_FILE = "fakenews_stemming_lemmatization.csv"

df = pd.read_csv(INPUT_FILE)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (2212, 7)


In [4]:
# 3. Select Text Column
if "body_text_preprocessed" in df.columns:
    TEXT_COLUMN = "body_text_preprocessed"

elif "headline_preprocessed" in df.columns:
    TEXT_COLUMN = "headline_preprocessed"

else:
    raise ValueError(
        "No preprocessed text column found. "
        "Run T7_text_preprocessing.py first."
    )

print("Selected text column:", TEXT_COLUMN)

Selected text column: body_text_preprocessed


In [5]:
# 4. Initialize NLP Tools
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

nlp = spacy.load("en_core_web_sm")

In [6]:

# 5. NLTK POS Mapping
def get_wordnet_pos(tag):

    if tag.startswith("J"):
        return wordnet.ADJ

    elif tag.startswith("V"):
        return wordnet.VERB

    elif tag.startswith("N"):
        return wordnet.NOUN

    elif tag.startswith("R"):
        return wordnet.ADV

    return wordnet.NOUN


In [7]:
# 6. Porter Stemming
def apply_stemming(text):

    if pd.isna(text):
        return ""

    words = re.findall(r"[a-z]+", str(text).lower())

    return " ".join(
        stemmer.stem(word) for word in words
    )


In [8]:
# 7. NLTK Lemmatization
def apply_nltk_lemmatization(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    words = word_tokenize(text)
    tagged_words = pos_tag(words)

    result = []

    for word, tag in tagged_words:

        if re.fullmatch(r"[a-z]+", word):

            pos = get_wordnet_pos(tag)

            lemma = lemmatizer.lemmatize(
                word,
                pos=pos
            )

            result.append(lemma)

    return " ".join(result)

In [9]:
# 8. spaCy Lemmatization

def apply_spacy_lemmatization(text):

    if pd.isna(text):
        return ""

    doc = nlp(str(text).lower())

    result = []

    for token in doc:

        if token.is_alpha:
            result.append(token.lemma_.lower())

    return " ".join(result)


In [10]:
# 9. Apply Transformations
print("\nApplying Porter stemming...")

df["stemmed_text"] = df[TEXT_COLUMN].apply(
    apply_stemming
)

print("Applying NLTK lemmatization...")

df["nltk_lemmatized_text"] = df[TEXT_COLUMN].apply(
    apply_nltk_lemmatization
)

print("Applying spaCy lemmatization...")

# nlp.pipe processes multiple documents efficiently.
# Process documents in batches instead of loading
# the entire dataset into one spaCy document.

texts = df[TEXT_COLUMN].fillna("").astype(str).tolist()

spacy_results = []

for doc in nlp.pipe(texts, batch_size=100):

    words = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha
    ]

    spacy_results.append(" ".join(words))

df["spacy_lemmatized_text"] = spacy_results


Applying Porter stemming...
Applying NLTK lemmatization...
Applying spaCy lemmatization...


In [11]:
# 10. Vocabulary Analysis
def get_vocabulary(texts):

    vocabulary = set()

    for text in texts:

        if pd.isna(text):
            continue

        words = re.findall(
            r"[a-z]+",
            str(text).lower()
        )

        vocabulary.update(words)

    return vocabulary


original_vocab = get_vocabulary(df[TEXT_COLUMN])

stemmed_vocab = get_vocabulary(df["stemmed_text"])

nltk_vocab = get_vocabulary(df["nltk_lemmatized_text"])

spacy_vocab = get_vocabulary(df["spacy_lemmatized_text"])


def vocabulary_reduction(original, transformed):

    if len(original) == 0:
        return 0.0

    return (
        (len(original) - len(transformed))
        / len(original)
    ) * 100

In [12]:

# 11. Compare Results

comparison = pd.DataFrame({
    "Technique": [
        "Original Text",
        "Porter Stemming",
        "NLTK Lemmatization",
        "spaCy Lemmatization"
    ],

    "Vocabulary Size": [
        len(original_vocab),
        len(stemmed_vocab),
        len(nltk_vocab),
        len(spacy_vocab)
    ]
})

original_size = len(original_vocab)

comparison["Vocabulary Reduction (%)"] = [
    0.0,
    vocabulary_reduction(original_vocab, stemmed_vocab),
    vocabulary_reduction(original_vocab, nltk_vocab),
    vocabulary_reduction(original_vocab, spacy_vocab)
]

print("\nVocabulary Comparison:")
print(comparison.to_string(index=False))


Vocabulary Comparison:
          Technique  Vocabulary Size  Vocabulary Reduction (%)
      Original Text               87                  0.000000
    Porter Stemming               85                  2.298851
 NLTK Lemmatization               80                  8.045977
spaCy Lemmatization               82                  5.747126


In [13]:





# 12. Display Example Results
print("\nExample Comparison:")

for index in range(min(3, len(df))):

    print("\nOriginal:")
    print(df[TEXT_COLUMN].iloc[index])

    print("\nPorter Stemmed:")
    print(df["stemmed_text"].iloc[index])

    print("\nNLTK Lemmatized:")
    print(df["nltk_lemmatized_text"].iloc[index])

    print("\nspaCy Lemmatized:")
    print(df["spacy_lemmatized_text"].iloc[index])


Example Comparison:

Original:
the prime minister attended the inauguration ceremony as the chief guest

Porter Stemmed:
the prime minist attend the inaugur ceremoni as the chief guest

NLTK Lemmatized:
the prime minister attend the inauguration ceremony a the chief guest

spaCy Lemmatized:
the prime minister attend the inauguration ceremony as the chief guest

Original:
government sources state that an official announcement will be made soon

Porter Stemmed:
govern sourc state that an offici announc will be made soon

NLTK Lemmatized:
government source state that an official announcement will be make soon

spaCy Lemmatized:
government source state that an official announcement will be make soon

Original:
local administration has visited the spot for further inspection

Porter Stemmed:
local administr ha visit the spot for further inspect

NLTK Lemmatized:
local administration have visit the spot for further inspection

spaCy Lemmatized:
local administration have visit the spot for f

In [14]:
# 13. Save Results
df.to_csv(OUTPUT_FILE, index=False)

comparison.to_csv(
    "T8_vocabulary_comparison.csv",
    index=False
)

print("\nProcessing completed successfully!")
print("Transformed dataset:", OUTPUT_FILE)
print("Vocabulary report: T8_vocabulary_comparison.csv")


Processing completed successfully!
Transformed dataset: fakenews_stemming_lemmatization.csv
Vocabulary report: T8_vocabulary_comparison.csv
